# 02 - Discovery and Slide Management

Validate layout discovery and slide CRUD/reorder operations from Phase 1 and Phase 2.

In [ ]:
import sys
from pathlib import Path
from pprint import pprint

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "python").exists() and (REPO_ROOT.parent / "python").exists():
    REPO_ROOT = REPO_ROOT.parent

PYTHON_ROOT = REPO_ROOT / "python"
if str(PYTHON_ROOT) not in sys.path:
    sys.path.insert(0, str(PYTHON_ROOT))


def build_service():
    from service import PowerPointService

    return PowerPointService


ARTIFACT_DIR = REPO_ROOT / "artifacts" / "notebook-tests"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
service = build_service()()
engine_info = service.dispatch("pptx_get_engine_info", {})
pprint(engine_info)

In [ ]:
create_result = service.dispatch("pptx_create_presentation", {"width": "10in", "height": "5.625in"})
presentation_id = create_result["presentation_id"]

layouts_result = service.dispatch("pptx_get_layouts", {"presentation_id": presentation_id})
masters_result = service.dispatch("pptx_get_masters", {"presentation_id": presentation_id})
theme_result = service.dispatch("pptx_get_theme", {"presentation_id": presentation_id})

layouts = layouts_result["layouts"]
layout_names = [layout["name"] for layout in layouts]
print("Layout count:", len(layouts))
print("Master count:", len(masters_result.get("masters", [])))
print("Theme keys:", sorted(theme_result.get("theme", {}).keys()))

preferred = ["Title and Content", "Title Slide", "Title Only", "Blank"]
layout_name = next((name for name in preferred if name in layout_names), layout_names[0])
print("Chosen layout:", layout_name)

layout_detail = service.dispatch(
    "pptx_get_layout_detail", {"presentation_id": presentation_id, "layout_name": layout_name}
)
print("Layout placeholder count:", layout_detail.get("placeholder_count"))

In [ ]:
add_1 = service.dispatch("pptx_add_slide", {"presentation_id": presentation_id, "layout_name": layout_name})
add_2 = service.dispatch(
    "pptx_add_slide", {"presentation_id": presentation_id, "layout_name": layout_name, "position": 1}
)

print("Added slides:", add_1.get("added_slide_index"), add_2.get("added_slide_index"))

state_before = service.dispatch("pptx_get_presentation_state", {"presentation_id": presentation_id})
print("Slide count before duplicate:", state_before["slide_count"])

dup = service.dispatch(
    "pptx_duplicate_slide",
    {
        "presentation_id": presentation_id,
        "source_index": 1,
        "target_position": 2,
    },
)
print("Duplicated index:", dup.get("duplicated_slide_index"))

move = service.dispatch("pptx_move_slide", {"presentation_id": presentation_id, "from_index": 2, "to_index": 1})
print("Moved:", move.get("from_index"), "->", move.get("to_index"))

In [ ]:
state = service.dispatch("pptx_get_presentation_state", {"presentation_id": presentation_id})
count = state["slide_count"]
new_order = list(range(count, 0, -1))
reorder = service.dispatch("pptx_reorder_slides", {"presentation_id": presentation_id, "new_order": new_order})
print("Reordered to:", reorder.get("new_order"))

if count >= 2:
    delete = service.dispatch("pptx_delete_slide", {"presentation_id": presentation_id, "slide_index": count})
    print("Deleted slide index:", delete.get("deleted_slide_index"))

final_state = service.dispatch("pptx_get_presentation_state", {"presentation_id": presentation_id})
print("Final slide count:", final_state["slide_count"])

In [ ]:
output_path = (ARTIFACT_DIR / "nb02_discovery_slides_output.pptx").resolve()
service.dispatch("pptx_save_presentation", {"presentation_id": presentation_id, "output_path": str(output_path)})
service.dispatch("pptx_close_presentation", {"presentation_id": presentation_id})
service.shutdown()
print("Saved:", output_path)